## Importing necessary libraries and functions

In [68]:
from tomograph_functions import radon_transform,back_projection,filtr_sinogram
from helpers import normalize_img,show_img
from dicom_functions import save_dicom_file,read_dicom_file
import os
import ipywidgets.widgets as widgets
import datetime

import ipywidgets as widgets
from IPython.display import display
from IPython.display import display
import warnings
#Supressing warrnigns
warnings.filterwarnings("ignore")

## Selecting the input image and set options of simulation

In [2]:
available_images=[file.name for file in os.scandir("./tomograf-dicom") if file.is_file()]

detectors=widgets.IntSlider(
    value=180,
    min=2,
    max=360,
    step=1,
    description='Detectors:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    layout=widgets.Layout(width='auto')
)
detectors_spread=widgets.FloatSlider(
    value=90.0,
    min=45.0,
    max=270,
    step=0.5,
    description='Angular spread:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
    layout=widgets.Layout(width='auto')
)

step=widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=10.0,
    step=0.1,
    description='Step value:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
    layout=widgets.Layout(width='auto')
)

cbox=widgets.Combobox(

    placeholder='Choose an image',
    options=available_images,
    description='Combobox:',
    ensure_option=True,
    disabled=False,
    layout=widgets.Layout(width='auto')
)
filtering=widgets.Checkbox(
    value=True,
    description='Use filtering',
    disabled=False,
    layout=widgets.Layout(width='auto')
)
show_steps=widgets.Checkbox(
    value=False,
    description='Show steps',
    disabled=False,
    layout=widgets.Layout(width='auto')
)
sliders=widgets.VBox([detectors,detectors_spread,step,cbox,filtering,show_steps],layout=widgets.Layout(width='45%'))
display(sliders)

### Displaying patient data

In [31]:
if cbox.value != "":
    image,data = read_dicom_file(f"tomograf-dicom/{cbox.value}")
    #PARAMETERS:list[str]=["Name","ID","Sex","Age","Study Date","Comments"]
    widgets_to_show:list=[]
    for parameter,value in data.items():
        text=value if value!="" else "No data "
        
        if "^" in text:
            text=" ".join(text.split("^"))
        if "Comments" in parameter:
            widgets_to_show.append(widgets.Textarea(description=f"{parameter.replace("Image","")}:", value=f"{text}",disabled=True))
            continue
        widgets_to_show.append(widgets.Text(description=f"{parameter.replace("Patient","").replace("Study","Study ")}:", value=f"{text}",disabled=True))

    display(widgets.VBox(widgets_to_show,layout=widgets.Layout(width='85%')))

    

### Radon transform and back projection

In [ ]:
final=None

if cbox.value != "":
    height, width = image.shape

    intermediate_sinograms, sinogram = radon_transform(
        image, detectors.value, step.value, detectors_spread.value
    )

    filtered_str = ""
    if filtering.value:
        sinogram = filtr_sinogram(sinogram)
        filtered_str = " (with filtering)"
        intermediate_sinograms = [filtr_sinogram(part_sin) for part_sin in intermediate_sinograms]

    intermediate_reconstructed, final = back_projection(
        sinogram, height, width, detectors.value, step.value, detectors_spread.value
    )
    final=normalize_img(final)
    intermediate_reconstructed=[normalize_img(recon) for recon in intermediate_reconstructed]

    input_image_out = widgets.Output()
    with input_image_out:
        show_img(image, title="Input image")

    final_sinogram_out = widgets.Output()
    with final_sinogram_out:
        show_img(sinogram, title=f"Sinogram{filtered_str}")

    final_image_out = widgets.Output()
    with final_image_out:
        show_img(final, title=f"Final image{filtered_str}")

    images_box = widgets.HBox([input_image_out, final_sinogram_out, final_image_out])
    display(images_box)
    

In [ ]:
#Wrappers for show_img to make interactive dispaying
if show_steps.value:
    def show_sinogram(step: int) -> None:
        show_img(intermediate_sinograms[step], title=f"Sinogram {filtered_str} number {step}")

    def show_reconstructed(step:int)->None:
        show_img(intermediate_reconstructed[step], title=f"Reconstructed {filtered_str} number {step}")

    slider_sinogram = widgets.IntSlider(
        min=0,
        max=len(intermediate_sinograms) - 1,
        step=1,
        value=0
        #interval=100
    )

    slider_reconstructed = widgets.IntSlider(#widgets.Play(
        min=0,
        max=len(intermediate_reconstructed) - 1,
        step=1,
        value=0,
        interval=100
    )

    part_sinogram = widgets.interactive_output(show_sinogram, {'step': slider_sinogram})
    part_reconstructed=widgets.interactive_output(show_reconstructed,{'step': slider_reconstructed})


    sliders = widgets.HBox([slider_sinogram,slider_reconstructed])
    part_images=widgets.HBox([part_sinogram,part_reconstructed])
    combined=widgets.VBox([sliders,part_images])
    display(combined)

### Adding the examination data

In [84]:
if cbox.value != "" and final is not None:
        patient_name=widgets.Textarea(description="Name", placeholder="Enter last name and first name (separated by a space)",disabled=False)
        patient_id=widgets.Text(description="ID", placeholder="Enter patient id",disabled=False)

        patient_sex=widgets.Combobox(
                placeholder='Choose a gender',
                options=["M","O","F"],
                description="Sex",
                ensure_option=True,
                disabled=False,
                layout=widgets.Layout(width='auto')
        )

        age_format=widgets.Combobox(
                placeholder='Choose age format',
                options=["M","W","Y"],
                description="Age format",
                ensure_option=True,
                disabled=False,
                layout=widgets.Layout(width='auto')
        )
        patient_age=widgets.IntSlider(
                description="Age value",
                min=1,
                max=130,
                step=1,
                value=0
                )
        
        examination_date=widgets.DatePicker(
        description='Study Date',
        disabled=False,
        layout=widgets.Layout(width='auto')
        )
        comments=widgets.Textarea(description="Comments", placeholder="Enter Comments",disabled=False,layout=widgets.Layout(width='auto'))
    
        widgets_to_show=[patient_name,patient_id,patient_sex,age_format,patient_age,examination_date,comments]
        display(widgets.VBox(widgets_to_show,layout=widgets.Layout(width='25%')))

### Saving the examination data